# Preprocessing Pipeline — Federated IIoT Intrusion Detection System

**Project:** Evaluating Federated Learning for Intrusion Detection in Industrial IoT  
**Dataset:** CIC DataSense IIoT 2025 (10-second aggregation window)  
**Target:** 8-class multi-class classification (`label2`)  

---

## Purpose of this Notebook

This notebook implements the preprocessing pipeline that transforms the raw DataSense CSV into a
clean, model-ready dataset. The output feeds into both:
- The **centralized baseline** (upper-bound benchmark)
- The **federated partitioning** script (`partitioning.py`)

## Preprocessing Steps

| Step | Action | Rationale |
|------|--------|----------|
| 1 | Load raw CSV | Combined 10-sec aggregation file |
| 2 | Drop non-feature columns | Timestamps, redundant labels, MAC addresses |
| 3 | Select the 17 features from DataSense Table 7 | Lightweight model for FL edge devices |
| 4 | Verify data quality | Confirm no missing/infinite values in selected features |
| 5 | Log-transform skewed features | Reduce right-skew for features with \|skewness\| > 5 |
| 6 | Encode target labels + compute class weights | LabelEncoder for `label2`, inverse-frequency weights |
| 7 | Save processed dataset | Single CSV for partitioning and training |

**Important FL Note:** StandardScaler is NOT applied here. In the federated setting, each client
fits the scaler on its own local training split. For the centralized baseline, the scaler is fit
on the global training split only. This is handled downstream in `federated.py` and `baselines.py`.

---
## Step 0 — Imports and Configuration

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from scipy.stats import skew
import os
import json
import warnings
warnings.filterwarnings('ignore')

RAW_DATA_PATH   = "../data/processed/datasense_10sec_combined.csv"
OUTPUT_DIR      = "../data/processed/"
OUTPUT_FILENAME = "datasense_preprocessed.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration ready.")

Configuration ready.


---
## Step 1 — Load the Raw Data

In [5]:
df = pd.read_csv(RAW_DATA_PATH)

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumn dtypes:")
print(f"  Numeric (int/float): {df.select_dtypes(include='number').shape[1]}")
print(f"  Object (string):     {df.select_dtypes(include='object').shape[1]}")
print(f"\nFirst 3 rows:")
df.head(3)

Dataset shape: 30,030 rows × 94 columns

Column dtypes:
  Numeric (int/float): 71
  Object (string):     23

First 3 rows:


,device_name,device_mac,label_full,label1,label2,label3,label4,timestamp,timestamp_start,timestamp_end,...,network_time-delta_min,network_time-delta_std_deviation,network_ttl_avg,network_ttl_max,network_ttl_min,network_ttl_std_deviation,network_window-size_avg,network_window-size_max,network_window-size_min,network_window-size_std_deviation
0,edge1,dc:a6:32:dc:27:d4,attack_ddos_syn-flood-port-80_edge1,attack,ddos,syn-flood-port-80,ddos_syn-flood-port-80,2025-01-23T15:31:10.709000Z_2025-01-23T15:31:2...,2025-01-23T15:31:10.709000Z,2025-01-23T15:31:20.709000Z,...,2.600000e-08,0.000042,64.0,64.0,64.0,0.0,17587.532313,64240.0,0.0,28377.701703
1,edge1,dc:a6:32:dc:27:d4,attack_ddos_syn-flood-port-80_edge1,attack,ddos,syn-flood-port-80,ddos_syn-flood-port-80,2025-01-23T15:31:15.709000Z_2025-01-23T15:31:2...,2025-01-23T15:31:15.709000Z,2025-01-23T15:31:25.709000Z,...,2.500000e-08,0.000041,64.0,64.0,64.0,0.0,17259.307890,64240.0,0.0,28201.477525
2,edge1,dc:a6:32:dc:27:d4,attack_ddos_syn-flood-port-80_edge1,attack,ddos,syn-flood-port-80,ddos_syn-flood-port-80,2025-01-23T15:31:20.709000Z_2025-01-23T15:31:3...,2025-01-23T15:31:20.709000Z,2025-01-23T15:31:30.709000Z,...,2.500000e-08,0.000041,64.0,64.0,64.0,0.0,17185.878307,64240.0,0.0,28161.438527


---
## Step 2 — Separate Metadata, Labels, and Features

The columns are split into three groups:

- **`device_name`** → kept separately for FL partitioning (not a model feature)  
- **`label2`** → our 8-class target variable  
- **Everything else in the metadata/label group** → dropped entirely  

Columns dropped:
- `device_mac` — device identifier, not a feature
- `label_full`, `label1`, `label3`, `label4` — alternative label granularities we don't need
- `timestamp`, `timestamp_start`, `timestamp_end` — time identifiers, would cause data leakage

In [6]:
# ── Columns to drop (metadata + unused labels) ─────────────────────
COLS_TO_DROP = [
    'device_mac',
    'label_full',
    'label1',
    'label3',
    'label4',
    'timestamp',
    'timestamp_start',
    'timestamp_end',
]

# ── Preserve device_name for FL partitioning ───────────────────────
device_names = df['device_name'].copy()
print(f"Unique devices: {device_names.nunique()}")
print(f"Device distribution (top 10):")
print(device_names.value_counts().head(10))

# ── Preserve target label ─────────────────────────────────────────
target = df['label2'].copy()
print(f"\nTarget (label2) distribution:")
print(target.value_counts())

# ── Drop non-feature columns ──────────────────────────────────────
# Also drop device_name (saved separately) and label2 (saved separately)
cols_to_remove = COLS_TO_DROP + ['device_name', 'label2']
existing_to_remove = [c for c in cols_to_remove if c in df.columns]
df_features = df.drop(columns=existing_to_remove)

print(f"\nAfter dropping metadata/labels: {df_features.shape[1]} columns remain")

Unique devices: 38
Device distribution (top 10):
device_name
edge1             2037
mqtt-broker       2034
router            1092
ap                 955
wisenet-camera     923
steam-sensor       892
weather-sensor     873
yi-camera          871
motion-sensor      870
light-sensor       822
Name: count, dtype: int64

Target (label2) distribution:
label2
benign        13680
recon          6005
dos            3280
ddos           3234
mitm           1471
malware        1444
web             552
bruteforce      364
Name: count, dtype: int64

After dropping metadata/labels: 84 columns remain


---
## Step 3 — Select the 17 Features from DataSense Table 7

The DataSense paper (Table 7) identified 17 features through a hierarchical multi-stage
information-driven feature selection process (context grouping → sparse group-lasso → 
RRS scoring → NSGA-II genetic algorithm).

This is ideal for our FL study because:
1. **Lightweight** — 17 features vs 71+ reduces model size and communication cost
2. **Validated** — the paper showed these features maintain high detection accuracy
3. **Diverse** — they span 7 context groups, capturing different aspects of network behavior

**Column name mapping note:** Three Table 7 features ("IPs Dst", "MACs Src", "Ports All") 
are list-type strings in the raw data. We use their `_count` numeric equivalents, which
capture the same cardinality information without requiring string vectorization.

In [7]:
# ── Mapping: DataSense Table 7 name → actual CSV column name ──────
# This dictionary documents the exact mapping for reproducibility.

TABLE7_FEATURE_MAP = {
    # Table 7 Name              : CSV Column Name                    # Context Group
    'Messages Count'            : 'log_messages_count',              # Log Data Rate
    'Data Range Mean'           : 'log_data-ranges_avg',             # Log Data Stats
    'Data Types List'           : 'log_data-types_count',            # Log Data Stats  (count equiv.)
    'Fragmented Packets'        : 'network_fragmented-packets',      # Fragmentation
    'Packet Interval'           : 'network_interval-packets',        # Packet Traffic Rate
    'Packets All Count'         : 'network_packets_all_count',       # Packet Traffic Rate
    'IPs Dst'                   : 'network_ips_dst_count',           # Address Diversity (count equiv.)
    'IPs All Count'             : 'network_ips_all_count',           # Address Diversity
    'MACs Src'                  : 'network_macs_src_count',          # Address Diversity (count equiv.)
    'Packet Size Std. Dev.'     : 'network_packet-size_std_deviation', # Size Length
    'Ports All'                 : 'network_ports_all_count',         # Network Multiplexing (count equiv.)
    'Protocols All Count'       : 'network_protocols_all_count',     # Network Multiplexing
    'Time Delta Mean'           : 'network_time-delta_avg',          # Timing Control
    'TTL Mean'                  : 'network_ttl_avg',                 # Timing Control
    'Window Size Mean'          : 'network_window-size_avg',         # Timing Control
    'IP Flags Maximum'          : 'network_ip-flags_max',            # Header Flags
    'TCP PSH Flag Count'        : 'network_tcp-flags-psh_count',     # Header Flags
}

SELECTED_FEATURES = list(TABLE7_FEATURE_MAP.values())

# ── Verify all 17 features exist in the dataset ───────────────────
missing = [f for f in SELECTED_FEATURES if f not in df.columns]
if missing:
    print(f"ERROR — Features not found in dataset: {missing}")
    print(f"Available columns containing similar names:")
    for m in missing:
        key = m.split('_')[1] if '_' in m else m
        matches = [c for c in df.columns if key in c.lower()]
        print(f"  '{m}' → possible matches: {matches}")
else:
    print(f"All 17 features found in dataset.")

# ── Select features ───────────────────────────────────────────────
df_selected = df[SELECTED_FEATURES].copy()

print(f"\nSelected feature matrix shape: {df_selected.shape}")
print(f"\nFeature summary:")
df_selected.describe().round(3)

All 17 features found in dataset.

Selected feature matrix shape: (30030, 17)

Feature summary:


,log_messages_count,log_data-ranges_avg,log_data-types_count,network_fragmented-packets,network_interval-packets,network_packets_all_count,network_ips_dst_count,network_ips_all_count,network_macs_src_count,network_packet-size_std_deviation,network_ports_all_count,network_protocols_all_count,network_time-delta_avg,network_ttl_avg,network_window-size_avg,network_ip-flags_max,network_tcp-flags-psh_count
count,30030.000,30030.000,30030.000,30030.000,30030.000,30030.000,30030.000,30030.000,30030.000,30030.000,30030.000,30030.000,30030.000,30030.000,30030.000,30030.000,30030.000
mean,5.139,67.915,0.401,3481.795,494.565,99590.638,10.637,10.944,5.718,99.004,6954.699,3.279,0.010,125.819,13220.514,1.434,10935.695
std,13.896,230.829,0.643,14728.429,901.932,273078.482,34.501,34.472,8.863,180.706,18881.050,2.377,0.014,75.690,16415.023,0.880,93855.510
min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,0.000,0.000,0.000,0.000,0.231,5.000,3.000,3.000,2.000,1.203,2.000,2.000,0.000,64.000,0.000,0.000,0.000
50%,0.000,0.000,0.000,0.000,30.992,56.000,3.000,3.000,3.000,12.923,4.000,3.000,0.006,127.667,3412.808,2.000,1.000
75%,2.000,0.000,1.000,0.000,710.115,8133.500,4.000,4.000,4.000,75.444,396.750,4.000,0.015,159.500,32522.824,2.000,12.000
max,206.000,1951.244,2.000,69251.000,9471.000,1487727.000,260.000,260.000,45.000,1968.558,65916.000,34.000,0.213,255.000,65535.000,2.000,1144412.000


---
## Step 4 — Data Quality Check on Selected Features

Even though the full EDA found no issues, we verify the 17 selected features specifically
for missing values, infinite values, and confirm all are numeric.

In [8]:
print("Data quality check on 17 selected features:")
print("─" * 50)

# Missing values
n_missing = df_selected.isnull().sum().sum()
print(f"Missing values:  {n_missing}  {'✓' if n_missing == 0 else '✗ ACTION NEEDED'}")

# Infinite values
n_inf = np.isinf(df_selected.select_dtypes(include='number')).sum().sum()
print(f"Infinite values: {n_inf}  {'✓' if n_inf == 0 else '✗ ACTION NEEDED'}")

# All numeric check
non_numeric = df_selected.select_dtypes(exclude='number').columns.tolist()
print(f"Non-numeric cols: {non_numeric if non_numeric else 'None ✓'}")

# Constant columns (zero variance)
constant_cols = [c for c in df_selected.columns if df_selected[c].nunique() <= 1]
print(f"Constant cols:   {constant_cols if constant_cols else 'None ✓'}")

# Data types
print(f"\nDtypes:")
print(df_selected.dtypes.value_counts())

Data quality check on 17 selected features:
──────────────────────────────────────────────────
Missing values:  0  ✓
Infinite values: 0  ✓
Non-numeric cols: None ✓
Constant cols:   None ✓

Dtypes:
int64      10
float64     7
Name: count, dtype: int64


---
## Step 5 — Log-Transform Skewed Features

Several features are heavily right-skewed (long tail of large values). This is common for
count-based network features where most time windows are quiet but a few have bursts.

We apply `log1p(x) = ln(1 + x)` to features with |skewness| > 5. The `+1` handles zeros safely.

**Why before scaling?** Log-transform reduces extreme skew, which makes the subsequent
StandardScaler (applied per-client during FL training) more effective — a scaler on heavily
skewed data is dominated by outliers.

In [9]:
SKEW_THRESHOLD = 5.0

# ── Compute skewness for each feature ─────────────────────────────
skewness = df_selected.apply(lambda col: skew(col, nan_policy='omit'))
skew_df = pd.DataFrame({
    'feature': skewness.index,
    'skewness': skewness.values,
    'abs_skewness': np.abs(skewness.values)
}).sort_values('abs_skewness', ascending=False)

print("Skewness of selected features (sorted by |skew|):")
print("─" * 60)
for _, row in skew_df.iterrows():
    flag = " ← TRANSFORM" if row['abs_skewness'] > SKEW_THRESHOLD else ""
    print(f"  {row['feature']:45s}  skew = {row['skewness']:>10.2f}{flag}")

# ── Apply log1p to heavily skewed features ────────────────────────
cols_to_transform = skew_df[skew_df['abs_skewness'] > SKEW_THRESHOLD]['feature'].tolist()
print(f"\nApplying log1p to {len(cols_to_transform)} features: {cols_to_transform}")

for col in cols_to_transform:
    before_skew = skew(df_selected[col])
    df_selected[col] = np.log1p(df_selected[col])
    after_skew = skew(df_selected[col])
    print(f"  {col}: skew {before_skew:.2f} → {after_skew:.2f}")

Skewness of selected features (sorted by |skew|):
────────────────────────────────────────────────────────────
  network_tcp-flags-psh_count                    skew =      10.20 ← TRANSFORM
  network_ips_dst_count                          skew =       6.00 ← TRANSFORM
  network_ips_all_count                          skew =       5.99 ← TRANSFORM
  log_messages_count                             skew =       4.61
  network_protocols_all_count                    skew =       4.31
  network_fragmented-packets                     skew =       4.06
  log_data-ranges_avg                            skew =       3.74
  network_packets_all_count                      skew =       3.47
  network_interval-packets                       skew =       3.40
  network_macs_src_count                         skew =       3.33
  network_time-delta_avg                         skew =       3.17
  network_ports_all_count                        skew =       2.63
  network_packet-size_std_deviation              

In [ ]:
# ── Save the list of transformed columns for documentation ────────
# This is needed so the same transform is applied at inference time.

transform_log = {
    'log1p_transformed_features': cols_to_transform,
    'skew_threshold': SKEW_THRESHOLD,
    'all_selected_features': SELECTED_FEATURES,
    'table7_feature_map': TABLE7_FEATURE_MAP,
}

log_path = os.path.join(OUTPUT_DIR, 'preprocessing_config.json')
with open(log_path, 'w') as f:
    json.dump(transform_log, f, indent=2)
print(f"Preprocessing config saved to {log_path}")

---
## Step 6 — Encode Target Labels and Compute Class Weights

### Label Encoding
Convert the 8 string class names in `label2` to integers 0–7 using `LabelEncoder`.

### Class Weights
The dataset has significant class imbalance (37.6:1 ratio between `benign` and `bruteforce`).
We compute **inverse-frequency weights** to pass to `CrossEntropyLoss` during training:

$$w_c = \frac{N}{C \times n_c}$$

where $N$ = total samples, $C$ = number of classes, $n_c$ = samples in class $c$.

**Why class weights instead of oversampling (SMOTE)?**  
In federated learning, each client has a small local dataset. SMOTE on tiny per-client
splits would generate synthetic samples from very few real examples, likely producing
noisy/unreliable features. Class weighting in the loss function is simpler, more robust,
and doesn't inflate the local dataset size (which would distort the FedAvg weighted aggregation).

In [10]:
# ── Encode labels ─────────────────────────────────────────────────
le = LabelEncoder()
encoded_labels = le.fit_transform(target)

print("Label encoding mapping:")
print("─" * 40)
for idx, class_name in enumerate(le.classes_):
    count = (encoded_labels == idx).sum()
    pct = count / len(encoded_labels) * 100
    print(f"  {idx} → {class_name:15s}  ({count:>6,} samples, {pct:5.2f}%)")

# ── Compute inverse-frequency class weights ───────────────────────
n_samples = len(encoded_labels)
n_classes = len(le.classes_)
class_counts = np.bincount(encoded_labels)

class_weights = n_samples / (n_classes * class_counts)

print(f"\nClass weights (inverse frequency):")
print("─" * 40)
for idx, class_name in enumerate(le.classes_):
    print(f"  {class_name:15s}  weight = {class_weights[idx]:.4f}")

# ── Save class weights and label mapping ──────────────────────────
label_config = {
    'label_mapping': {int(idx): name for idx, name in enumerate(le.classes_)},
    'class_weights': {int(idx): float(w) for idx, w in enumerate(class_weights)},
    'n_classes': n_classes,
}

label_path = os.path.join(OUTPUT_DIR, 'label_config.json')
with open(label_path, 'w') as f:
    json.dump(label_config, f, indent=2)
print(f"\nLabel config saved to {label_path}")

Label encoding mapping:
────────────────────────────────────────
  0 → benign           (13,680 samples, 45.55%)
  1 → bruteforce       (   364 samples,  1.21%)
  2 → ddos             ( 3,234 samples, 10.77%)
  3 → dos              ( 3,280 samples, 10.92%)
  4 → malware          ( 1,444 samples,  4.81%)
  5 → mitm             ( 1,471 samples,  4.90%)
  6 → recon            ( 6,005 samples, 20.00%)
  7 → web              (   552 samples,  1.84%)

Class weights (inverse frequency):
────────────────────────────────────────
  benign           weight = 0.2744
  bruteforce       weight = 10.3125
  ddos             weight = 1.1607
  dos              weight = 1.1444
  malware          weight = 2.5995
  mitm             weight = 2.5518
  recon            weight = 0.6251
  web              weight = 6.8003

Label config saved to ../data/processed/label_config.json


---
## Step 7 — Assemble and Save the Processed Dataset

The final CSV contains:
- **17 feature columns** (log1p-transformed where applicable)
- **`attack_category`** — integer-encoded 8-class target
- **`device_name`** — preserved for FL partitioning by device

**What is NOT in this file:**
- StandardScaler is NOT applied — this happens per-client (FL) or on train split (centralized)
- No train/test split — partitioning handles this downstream

This design keeps the preprocessing pipeline clean and ensures each downstream
consumer (centralized baseline, IID FL, non-IID FL) can apply scaling correctly.

In [11]:
# ── Assemble final DataFrame ──────────────────────────────────────
df_processed = df_selected.copy()
df_processed['attack_category'] = encoded_labels
df_processed['device_name'] = device_names.values

# ── Save ──────────────────────────────────────────────────────────
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)
df_processed.to_csv(output_path, index=False)

print(f"Processed dataset saved to: {output_path}")
print(f"Shape: {df_processed.shape[0]:,} rows × {df_processed.shape[1]} columns")
print(f"  → 17 feature columns")
print(f"  → 1 target column (attack_category)")
print(f"  → 1 metadata column (device_name)")
print(f"\nFile size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

Processed dataset saved to: ../data/processed/datasense_preprocessed.csv
Shape: 30,030 rows × 19 columns
  → 17 feature columns
  → 1 target column (attack_category)
  → 1 metadata column (device_name)

File size: 4.65 MB


In [12]:
# ── Final verification ────────────────────────────────────────────
df_check = pd.read_csv(output_path)

print("Final dataset verification:")
print("═" * 60)
print(f"Rows:              {df_check.shape[0]:,}")
print(f"Columns:           {df_check.shape[1]}")
print(f"Missing values:    {df_check.isnull().sum().sum()}")
print(f"Feature columns:   {[c for c in df_check.columns if c not in ['attack_category', 'device_name']]}")
print(f"Target classes:    {sorted(df_check['attack_category'].unique())}")
print(f"Devices:           {df_check['device_name'].nunique()}")
print(f"\nTarget distribution:")
print(df_check['attack_category'].value_counts().sort_index())
print(f"\nFirst 5 rows:")
df_check.head()

Final dataset verification:
════════════════════════════════════════════════════════════
Rows:              30,030
Columns:           19
Missing values:    0
Feature columns:   ['log_messages_count', 'log_data-ranges_avg', 'log_data-types_count', 'network_fragmented-packets', 'network_interval-packets', 'network_packets_all_count', 'network_ips_dst_count', 'network_ips_all_count', 'network_macs_src_count', 'network_packet-size_std_deviation', 'network_ports_all_count', 'network_protocols_all_count', 'network_time-delta_avg', 'network_ttl_avg', 'network_window-size_avg', 'network_ip-flags_max', 'network_tcp-flags-psh_count']
Target classes:    [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Devices:           38

Target distribution:
attack_category
0    13680
1      364
2     3234
3     3280
4     1444
5     1471
6     6005
7      552
Name: count, dtype: int64

First 5 rows:


,log_messages_count,log_data-ranges_avg,log_data-types_count,network_fragmented-packets,network_interval-packets,network_packets_all_count,network_ips_dst_count,network_ips_all_count,network_macs_src_count,network_packet-size_std_deviation,network_ports_all_count,network_protocols_all_count,network_time-delta_avg,network_ttl_avg,network_window-size_avg,network_ip-flags_max,network_tcp-flags-psh_count,attack_category,device_name
0,0,0.0,0,0,0.007033,161243,2.079442,2.079442,7,0.0,24201,2,0.000007,64.0,17587.532313,2.0,0.0,2,edge1
1,0,0.0,0,0,0.006824,905009,2.079442,2.079442,7,0.0,65916,2,0.000007,64.0,17259.307890,2.0,0.0,2,edge1
2,0,0.0,0,0,0.006722,1487640,1.945910,1.945910,6,0.0,65916,2,0.000007,64.0,17185.878307,2.0,0.0,2,edge1
3,0,0.0,0,0,0.006722,1487609,2.079442,2.079442,6,0.0,65916,2,0.000007,64.0,16750.718465,2.0,0.0,2,edge1
4,0,0.0,0,0,0.006722,1487671,2.079442,2.079442,6,0.0,65916,2,0.000007,64.0,16673.446073,2.0,0.0,2,edge1


---
## Summary

### What was done
1. Loaded raw DataSense CSV (30,030 rows × 94 columns)
2. Dropped 10 non-feature columns (timestamps, MAC, unused labels)
3. Selected 17 features from Table 7 (mapping list-type features to count equivalents)
4. Verified data quality — no missing, infinite, or constant values
5. Applied `log1p` transform to features with |skewness| > 5
6. Encoded `label2` → `attack_category` (0–7) with inverse-frequency class weights
7. Saved processed dataset with `device_name` preserved for FL partitioning

### Output files
| File | Purpose |
|------|--------|
| `data/processed/datasense_preprocessed.csv` | Model-ready dataset (17 features + target + device_name) |
| `data/processed/preprocessing_config.json` | Feature list, log1p columns, Table 7 mapping |
| `data/processed/label_config.json` | Label encoding, class weights |

### Next step
→ **Partitioning** (`partitioning.py`): Create IID and non-IID federated splits using `device_name` and `attack_category`.